In [1]:
import random, os, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

PREPROCESSED_DIR = "/kaggle/input/datasets/iftekharuddin27/preprocessed-datasets"
OUTPUT_DIR       = "/kaggle/working/"

MODEL_EN   = "distilbert-base-uncased"
MODEL_BN   = "distilbert-base-multilingual-cased"
MAX_LEN    = 128
BATCH_SIZE = 64     
EPOCHS     = 5
LR         = 3e-5  
ALPHA, BETA, GAMMA = 0.3, 0.3, 0.4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = {0: 'Non-hateful', 1: 'Hateful', 2: 'Sarcastic'}
print(f"Device: {device}")

Device: cuda


In [2]:
en_train = pd.read_csv(f"{PREPROCESSED_DIR}/en_train.csv")
en_val   = pd.read_csv(f"{PREPROCESSED_DIR}/en_val.csv")
en_test  = pd.read_csv(f"{PREPROCESSED_DIR}/en_test.csv")
bn_train = pd.read_csv(f"{PREPROCESSED_DIR}/bn_train.csv")
bn_val   = pd.read_csv(f"{PREPROCESSED_DIR}/bn_val.csv")
bn_test  = pd.read_csv(f"{PREPROCESSED_DIR}/bn_test.csv")

for df in [en_train, en_val, en_test, bn_train, bn_val, bn_test]:
    df['text_clean'] = df['text_clean'].fillna('')

def add_aux_labels(df, label_col):
    df['is_hateful']   = (df[label_col] == 1).astype(int)
    df['is_sarcastic'] = (df[label_col] == 2).astype(int)
    return df

en_train = add_aux_labels(en_train, 'class')
en_val   = add_aux_labels(en_val,   'class')
en_test  = add_aux_labels(en_test,  'class')
bn_train = add_aux_labels(bn_train, 'label')
bn_val   = add_aux_labels(bn_val,   'label')
bn_test  = add_aux_labels(bn_test,  'label')

print(f"English — Train:{len(en_train):,} Val:{len(en_val):,} Test:{len(en_test):,}")
print(f"Bangla  — Train:{len(bn_train):,} Val:{len(bn_val):,} Test:{len(bn_test):,}")

English — Train:83,455 Val:10,432 Test:10,432
Bangla  — Train:67,009 Val:8,376 Test:8,377


In [3]:
class DualHeadDataset(Dataset):
    def __init__(self, texts, labels_3class, labels_hate,
                 labels_sarc, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_len, return_tensors='pt'
        )
        self.labels_3class = torch.tensor(labels_3class, dtype=torch.long)
        self.labels_hate   = torch.tensor(labels_hate,   dtype=torch.float)
        self.labels_sarc   = torch.tensor(labels_sarc,   dtype=torch.float)

    def __len__(self): return len(self.labels_3class)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels_3class'] = self.labels_3class[idx]
        item['labels_hate']   = self.labels_hate[idx]
        item['labels_sarc']   = self.labels_sarc[idx]
        return item

In [4]:
# ============================================================
#  DistilBERT Dual-Head Model 
# ============================================================
class LightweightDualHeadModel(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout=0.3):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size  

        self.dropout = nn.Dropout(dropout)

        # Hate head 
        self.hate_head = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

        # Sarcasm head
        self.sarcasm_head = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

        # Fusion MLP (lighter)
        self.fusion_mlp = nn.Sequential(
            nn.Linear(hidden_size + 128 + 128, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        cls_output = self.dropout(outputs.last_hidden_state[:, 0, :])

        hate_hidden = F.relu(self.hate_head[0](cls_output))
        hate_hidden = self.dropout(hate_hidden)
        hate_logit  = self.hate_head[3](hate_hidden)

        sarc_hidden = F.relu(self.sarcasm_head[0](cls_output))
        sarc_hidden = self.dropout(sarc_hidden)
        sarc_logit  = self.sarcasm_head[3](sarc_hidden)

        fused         = torch.cat([cls_output, hate_hidden, sarc_hidden], dim=1)
        logits_3class = self.fusion_mlp(fused)

        return logits_3class, hate_logit.squeeze(-1), sarc_logit.squeeze(-1)

In [5]:
# =============================
#  Train / Eval 
# =============================
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    crit_3 = nn.CrossEntropyLoss()
    crit_b = nn.BCEWithLogitsLoss()

    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        l3    = batch['labels_3class'].to(device)
        lh    = batch['labels_hate'].to(device)
        ls    = batch['labels_sarc'].to(device)

        optimizer.zero_grad()
        logits, hl, sl = model(ids, mask)
        loss = ALPHA*crit_b(hl,ls) + BETA*crit_b(sl,ls) + GAMMA*crit_3(logits,l3)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device, return_preds=False):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            logits, _, _ = model(ids, mask)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels_3class'].numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')
    if return_preds:
        return f1, np.array(all_preds), np.array(all_labels)
    return f1

In [6]:
# ============================================================
# Train English with DistilBERT backbone
# ============================================================
print("="*65)
print("Training Lightweight Dual-Head (DistilBERT) — English")
print("="*65)

tok_en = AutoTokenizer.from_pretrained(MODEL_EN)

train_ds = DualHeadDataset(en_train['text_clean'].values, en_train['class'].values,
    en_train['is_hateful'].values, en_train['is_sarcastic'].values, tok_en)
val_ds   = DualHeadDataset(en_val['text_clean'].values, en_val['class'].values,
    en_val['is_hateful'].values, en_val['is_sarcastic'].values, tok_en)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2)

model_en  = LightweightDualHeadModel(MODEL_EN).to(device)
optimizer = torch.optim.AdamW(model_en.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, len(train_loader)//10, len(train_loader)*EPOCHS)

best_f1, best_state, hist_en = 0, None, []

for epoch in range(EPOCHS):
    loss = train_epoch(model_en, train_loader, optimizer, scheduler, device)
    f1   = evaluate(model_en, val_loader, device)
    hist_en.append({'epoch': epoch+1, 'loss': loss, 'val_f1': f1})
    if f1 > best_f1:
        best_f1   = f1
        best_state = {k: v.clone() for k, v in model_en.state_dict().items()}
    print(f"  Epoch {epoch+1}/{EPOCHS} — Loss: {loss:.4f} — Val F1: {f1:.4f}")

model_en.load_state_dict(best_state)
print(f"\nBest Val F1 (English): {best_f1:.4f}")

os.makedirs(f"{OUTPUT_DIR}lightweight_en", exist_ok=True)
torch.save(model_en.state_dict(), f"{OUTPUT_DIR}lightweight_en/model.pt")
tok_en.save_pretrained(f"{OUTPUT_DIR}lightweight_en")

Training Lightweight Dual-Head (DistilBERT) — English


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 1/5 — Loss: 0.3360 — Val F1: 0.8730
  Epoch 2/5 — Loss: 0.1803 — Val F1: 0.8823
  Epoch 3/5 — Loss: 0.1149 — Val F1: 0.8878
  Epoch 4/5 — Loss: 0.0768 — Val F1: 0.8880
  Epoch 5/5 — Loss: 0.0551 — Val F1: 0.8889

Best Val F1 (English): 0.8889


('/kaggle/working/lightweight_en/tokenizer_config.json',
 '/kaggle/working/lightweight_en/tokenizer.json')

In [7]:
# ============================================================
# Train Bangla with multilingual DistilBERT
# ============================================================
print("="*65)
print("Training Lightweight Dual-Head (mDistilBERT) — Bangla")
print("="*65)

from sklearn.utils.class_weight import compute_class_weight
cw_bn = compute_class_weight('balanced',
        classes=np.array([0,1,2]), y=bn_train['label'].values)
print(f"Bangla class weights: {cw_bn}")

tok_bn = AutoTokenizer.from_pretrained(MODEL_BN)

train_ds_bn = DualHeadDataset(bn_train['text_clean'].values, bn_train['label'].values,
    bn_train['is_hateful'].values, bn_train['is_sarcastic'].values, tok_bn)
val_ds_bn   = DualHeadDataset(bn_val['text_clean'].values, bn_val['label'].values,
    bn_val['is_hateful'].values, bn_val['is_sarcastic'].values, tok_bn)

train_loader_bn = DataLoader(train_ds_bn, batch_size=BATCH_SIZE, shuffle=True)
val_loader_bn   = DataLoader(val_ds_bn,   batch_size=BATCH_SIZE*2)

model_bn  = LightweightDualHeadModel(MODEL_BN).to(device)
optimizer = torch.optim.AdamW(model_bn.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, len(train_loader_bn)//10, len(train_loader_bn)*EPOCHS)

best_f1_bn, best_state_bn, hist_bn = 0, None, []

for epoch in range(EPOCHS):
    loss = train_epoch(model_bn, train_loader_bn, optimizer, scheduler, device)
    f1   = evaluate(model_bn, val_loader_bn, device)
    hist_bn.append({'epoch': epoch+1, 'loss': loss, 'val_f1': f1})
    if f1 > best_f1_bn:
        best_f1_bn   = f1
        best_state_bn = {k: v.clone() for k, v in model_bn.state_dict().items()}
    print(f"  Epoch {epoch+1}/{EPOCHS} — Loss: {loss:.4f} — Val F1: {f1:.4f}")

model_bn.load_state_dict(best_state_bn)
print(f"\nBest Val F1 (Bangla): {best_f1_bn:.4f}")

os.makedirs(f"{OUTPUT_DIR}lightweight_bn", exist_ok=True)
torch.save(model_bn.state_dict(), f"{OUTPUT_DIR}lightweight_bn/model.pt")
tok_bn.save_pretrained(f"{OUTPUT_DIR}lightweight_bn")

Training Lightweight Dual-Head (mDistilBERT) — Bangla
Bangla class weights: [0.67442656 1.09080106 1.66527498]


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 1/5 — Loss: 0.4873 — Val F1: 0.7367
  Epoch 2/5 — Loss: 0.3679 — Val F1: 0.7545
  Epoch 3/5 — Loss: 0.3144 — Val F1: 0.7784
  Epoch 4/5 — Loss: 0.2657 — Val F1: 0.7803
  Epoch 5/5 — Loss: 0.2275 — Val F1: 0.7867

Best Val F1 (Bangla): 0.7867


('/kaggle/working/lightweight_bn/tokenizer_config.json',
 '/kaggle/working/lightweight_bn/tokenizer.json')

In [9]:
# ========================
#  Test evaluation 
# ========================
def test_eval(model, tokenizer, test_df, label_col,
              model_str, lang, is_distilbert=True):
    test_ds = DualHeadDataset(
        test_df['text_clean'].values, test_df[label_col].values,
        test_df['is_hateful'].values, test_df['is_sarcastic'].values,
        tokenizer, MAX_LEN)
    loader = DataLoader(test_ds, batch_size=64, shuffle=False)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            logits, _, _ = model(ids, mask)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(batch['labels_3class'].numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')

    print(f"\n{'='*60}")
    print(f"{model_str} [TEST] — {lang}")
    print(f"{'='*60}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {f1:.4f}")
    print(classification_report(all_labels, all_preds,
                                target_names=list(CLASS_NAMES.values())))
    np.save(f"{OUTPUT_DIR}test_preds_{model_str}_{lang}.npy",
            np.array(all_preds))
    return acc, f1

acc_en, f1_en = test_eval(model_en, tok_en, en_test, 'class',
                           'Lightweight_DualHead', 'English')
acc_bn, f1_bn = test_eval(model_bn, tok_bn, bn_test, 'label',
                           'Lightweight_DualHead', 'Bangla')


print("\n" + "="*60)
print("PHASE 6B COMPARISON TABLE")
print("="*60)
comp = pd.DataFrame({
    'Model':  ['BERT baseline', 'DualHead XLM-R',
               'BanglaBERT baseline', 'DualHead XLM-R',
               'Lightweight DistilBERT', 'Lightweight mDistilBERT'],
    'Lang':   ['EN','EN','BN','BN','EN','BN'],
    'Test F1':[ 0.9029, '0.8983', 0.8200, '0.8151',
                round(f1_en,4), round(f1_bn,4)]
})
print(comp.to_string(index=False))
comp.to_csv(f"{OUTPUT_DIR}phase6b_results.csv", index=False)


Lightweight_DualHead [TEST] — English
Accuracy : 0.8925
Macro F1 : 0.8927
              precision    recall  f1-score   support

 Non-hateful       0.85      0.87      0.86      3470
     Hateful       0.92      0.91      0.91      3486
   Sarcastic       0.91      0.90      0.91      3476

    accuracy                           0.89     10432
   macro avg       0.89      0.89      0.89     10432
weighted avg       0.89      0.89      0.89     10432


Lightweight_DualHead [TEST] — Bangla
Accuracy : 0.8116
Macro F1 : 0.7919
              precision    recall  f1-score   support

 Non-hateful       0.83      0.81      0.82      4140
     Hateful       0.87      0.91      0.89      2560
   Sarcastic       0.68      0.65      0.66      1677

    accuracy                           0.81      8377
   macro avg       0.79      0.79      0.79      8377
weighted avg       0.81      0.81      0.81      8377


PHASE 6B COMPARISON TABLE
                  Model Lang Test F1
          BERT baseline  